# Grid Search: Window-Start × Favorite-Entry-Price (Winners Only)

Sweeps two variables across every recorded 15-min contract window:
1. **window_start** – minutes before close when the bot begins watching (7.0 → 0.17 min, 10s steps)
2. **entry_price** – minimum favorite-side price to trigger entry (51¢ → 90¢, 1¢ steps)

Losses are ignored. Only winning settlements contribute to profit.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
import pandas as pd
import numpy as np
import time

# ── paths (Colab / Drive) ────────────────────────────────────────────
TICK_CSV = "/content/drive/MyDrive/commod15min_ticks.csv"
OUT_OVERALL = "/content/drive/MyDrive/grid_tp_overall.csv"
OUT_BY_ASSET = "/content/drive/MyDrive/grid_tp_by_asset.csv"
OUT_TOP_OVERALL = "/content/drive/MyDrive/grid_tp_top100.csv"
OUT_TOP_ASSET = "/content/drive/MyDrive/grid_tp_top20_per_asset.csv"

# ── parameters ────────────────────────────────────────────────────────
LOTS = 10
SETTLE_WINDOW_SEC = 30.0

WINDOW_STEP_SEC = 10.0
window_grid = np.arange(
    WINDOW_STEP_SEC / 60.0,
    7.0 + WINDOW_STEP_SEC / 60.0,
    WINDOW_STEP_SEC / 60.0,
)[::-1]  # 7.0, 6.83, ... 0.17  (minutes before close)

price_grid = np.arange(51.0, 91.0, 1.0)  # 51, 52, ..., 90 cents

print(f"window steps : {len(window_grid)}  ({window_grid[0]:.2f} -> {window_grid[-1]:.2f} min before close)")
print(f"price steps  : {len(price_grid)}  ({price_grid[0]:.0f}c -> {price_grid[-1]:.0f}c)")
print(f"total combos : {len(window_grid) * len(price_grid)}")

In [ ]:
def determine_settlement(minute, ymid):
    sec_to_close = (15.0 - minute) * 60.0
    final_mask = sec_to_close <= SETTLE_WINDOW_SEC
    if final_mask.any():
        return ymid[final_mask].mean() > 50.0
    return ymid[-1] > 50.0


def process_window(minute, ymid, yes_won):
    n = len(minute)
    if n == 0:
        return {}

    fav_price = np.maximum(ymid, 100.0 - ymid)
    fav_is_yes = ymid >= 50.0

    results = {}

    for ws in window_grid:
        open_minute = 15.0 - ws
        valid = minute >= open_minute
        if not valid.any():
            continue
        valid_idx = np.where(valid)[0]
        valid_fav = fav_price[valid_idx]

        for ep in price_grid:
            hits = valid_fav >= ep
            if not hits.any():
                continue
            first_hit = np.argmax(hits)
            entry_i = valid_idx[first_hit]
            actual_px = fav_price[entry_i]
            is_yes = fav_is_yes[entry_i]
            bot_won = (is_yes and yes_won) or (not is_yes and not yes_won)
            profit = (100.0 - actual_px) * LOTS if bot_won else 0.0
            results[(round(float(ws), 4), float(ep))] = (bot_won, profit)

    return results

In [ ]:
t0 = time.time()
print(f"loading {TICK_CSV} ...")
df = pd.read_csv(TICK_CSV, usecols=[
    "ticker", "asset", "yes_bid", "yes_ask", "yes_mid", "sec_to_close"
])
df = df.dropna(subset=["yes_mid", "sec_to_close"])
df["minute_in_cycle"] = 15.0 - df["sec_to_close"] / 60.0
df = df[(df["minute_in_cycle"] >= 0) & (df["minute_in_cycle"] <= 15)]

windows = []
for ticker, g in df.groupby("ticker", sort=False):
    g = g.sort_values("minute_in_cycle")
    minute = g["minute_in_cycle"].to_numpy()
    ymid = g["yes_mid"].to_numpy(dtype=float)
    asset = g["asset"].iloc[0]
    yes_won = determine_settlement(minute, ymid)
    windows.append((asset, minute, ymid, yes_won))

n_windows = len(windows)
print(f"loaded {n_windows} windows in {time.time()-t0:.1f}s")

In [ ]:
# ── main grid sweep ──────────────────────────────────────────────────
agg = {}  # (ws, ep, asset) -> [n_entries, n_wins, total_profit_c]
report_every = max(1, n_windows // 20)

for wi, (asset, minute, ymid, yes_won) in enumerate(windows):
    res = process_window(minute, ymid, yes_won)
    for (ws, ep), (won, profit) in res.items():
        key = (ws, ep, asset)
        if key not in agg:
            agg[key] = [0, 0, 0.0]
        agg[key][0] += 1
        if won:
            agg[key][1] += 1
            agg[key][2] += profit
    if (wi + 1) % report_every == 0:
        pct = 100 * (wi + 1) / n_windows
        print(f"  {wi+1}/{n_windows} ({pct:.0f}%) -- {time.time()-t0:.0f}s elapsed")

print(f"sweep done in {(time.time()-t0)/60:.1f} min")

In [ ]:
# ── build results ────────────────────────────────────────────────────
rows = []
for (ws, ep, asset), (n_ent, n_win, prof) in agg.items():
    rows.append({
        "window_min_before_close": ws,
        "entry_threshold_c": ep,
        "asset": asset,
        "n_entries": n_ent,
        "n_wins": n_win,
        "win_rate": round(n_win / n_ent, 4) if n_ent else 0,
        "total_profit_c": round(prof, 1),
        "avg_profit_per_entry_c": round(prof / n_ent, 2) if n_ent else 0,
    })
detail = pd.DataFrame(rows)
detail.to_csv(OUT_BY_ASSET, index=False)
print(f"wrote {OUT_BY_ASSET} ({len(detail)} rows)")

overall = (
    detail
    .groupby(["window_min_before_close", "entry_threshold_c"])
    .agg(
        n_entries=("n_entries", "sum"),
        n_wins=("n_wins", "sum"),
        total_profit_c=("total_profit_c", "sum"),
    )
    .reset_index()
)
overall["win_rate"] = (overall["n_wins"] / overall["n_entries"]).round(4)
overall["avg_profit_per_entry_c"] = (
    overall["total_profit_c"] / overall["n_entries"]
).round(2)
overall = overall.sort_values("total_profit_c", ascending=False)
overall.to_csv(OUT_OVERALL, index=False)
print(f"wrote {OUT_OVERALL} ({len(overall)} rows)")

overall.head(100).to_csv(OUT_TOP_OVERALL, index=False)
print(f"wrote {OUT_TOP_OVERALL}")

top_asset = (
    detail
    .sort_values("total_profit_c", ascending=False)
    .groupby("asset", group_keys=False)
    .head(20)
    .sort_values(["asset", "total_profit_c"], ascending=[True, False])
)
top_asset.to_csv(OUT_TOP_ASSET, index=False)
print(f"wrote {OUT_TOP_ASSET} ({len(top_asset)} rows)")

print(f"\ntotal time: {(time.time()-t0)/60:.1f} min")

In [ ]:
# ── display top results ──────────────────────────────────────────────
cols = ["window_min_before_close", "entry_threshold_c",
        "n_entries", "n_wins", "win_rate",
        "total_profit_c", "avg_profit_per_entry_c"]

print("TOP 15 COMBOS BY TOTAL PROFIT (all assets, winners only)")
print("=" * 70)
overall[cols].head(15)

In [ ]:
# ── top 5 per asset ──────────────────────────────────────────────────
assets = sorted(detail["asset"].unique())
print(f"TOP 5 PER ASSET  ({len(assets)} assets)")
print("=" * 70)
for a in assets:
    sub = (detail[detail["asset"] == a]
           .sort_values("total_profit_c", ascending=False))
    if sub.empty:
        continue
    print(f"\n  {a}:")
    display(sub[cols].head(5))